# Fourier Decomposition: fitting, uncertainty, and validation

This notebook is a server-portable workflow for:

1. single-source Fourier decomposition,
2. catalog multiprocessing,
3. conditional covariance and full-pipeline epoch/transit bootstrap,
4. phase-folded light curves with bootstrap confidence intervals, and
5. $\log P$--Fourier diagrams with measurement error bars.

In [ ]:
# -----------------------------------------------------------------------------
# 0. CPU / import environment
# -----------------------------------------------------------------------------
import os
import sys
from pathlib import Path

# Set before importing NumPy/SciPy when possible. One BLAS thread per worker
# avoids nested parallelism on a CPU server.
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

# Change only this path when the repository is installed elsewhere.
REPO_ROOT = Path.cwd().resolve()
SOURCE_DIR = REPO_ROOT / 'source'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

print('Repository:', REPO_ROOT)
print('Python:', sys.executable)


In [ ]:
# -----------------------------------------------------------------------------
# 1. Imports
# -----------------------------------------------------------------------------
import json
import multiprocessing as mp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from FourierDecomp import decomposition as decomp
from FourierDecomp import params
from FourierDecomp.IO import (
    build_fd_header, data_loader, epoch_arrays, get_data_config, wire_globals,
)
from FourierDecomp.LC import plot_lc_conditional_ci
from FourierDecomp.LSQ import F
from FourierDecomp.threading import mp_run
from FourierDecomp.uncertainty import (
    bootstrap_fourier_decomp, conditional_shared_invariant_uncertainty,
)


## 2. Configuration

All server-dependent paths are collected in the next cell. `INIT='lsq'` is the self-contained default. If RRFit initialization is required, load `df_rrfit` and `templates` before the wiring cell and set `INIT='rrfit'`.


In [ ]:
# -----------------------------------------------------------------------------
# 2.1 Paths and run controls
# -----------------------------------------------------------------------------
MODE = 'gaia'                       # 'gaia', 'ogle', or 'ztf'
IDENT_PATH = REPO_ROOT / 'data' / 'gaia_ident.ecsv'
PHOT_DIR = REPO_ROOT / 'data' / 'gaia_epoch_phot'
OGLE_IDENT_GLOB = 'data/ogle4/ident/*cep_ident*'
OUTPUT_DIR = REPO_ROOT / 'output' / 'reliability_workflow'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FD_OUTPUT = OUTPUT_DIR / f'fd_{MODE}_corrected.dat'
BOOTSTRAP_DIR = OUTPUT_DIR / f'bootstrap_{MODE}'
FIGURE_DIR = OUTPUT_DIR / 'figures'
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

INIT = 'lsq'
RANDOM_STATE = 20260827
N_FIT_WORKERS = 12
N_BOOT_WORKERS = 8
MP_CONTEXT = 'fork' if sys.platform != 'win32' else 'spawn'

# Safety switches: change explicitly after checking paths and a single fit.
RUN_CATALOG_FIT = False
RUN_SINGLE_BOOTSTRAP = False
RUN_BOOTSTRAP_BATCH = False

print('FD output:', FD_OUTPUT)
print('Bootstrap output:', BOOTSTRAP_DIR)


In [ ]:
# -----------------------------------------------------------------------------
# 2.2 Fourier fitting parameters
# -----------------------------------------------------------------------------
params.M_MIN = 3
params.M_MAX = 15
params.M_PAD = 2
params.ERR_FLOOR = 0.001
params.mode_default = MODE
params.coef_mode = 'ab'
params.quality_weight = True
params.opt_method = 'lasso'
params.PHASE_GAP_NONLINEAR_INIT = False

FIT_KWARGS = dict(
    init=INIT,
    period_fit=False,
    use_optim=True,
    adaptive_lam=True,
    use_refit=True,
)

print(FIT_KWARGS)


## 3. Load and wire data

`wire_globals` preserves the original package workflow. The bootstrap wrapper imports the same wired `decomposition` module, so no large data object is copied for each replicate.


In [ ]:
# -----------------------------------------------------------------------------
# 3.1 Load epoch photometry and identifiers
# -----------------------------------------------------------------------------
if MODE == 'ogle':
    ident_input = sorted(REPO_ROOT.glob(OGLE_IDENT_GLOB))
    if not ident_input:
        raise FileNotFoundError(f'No OGLE identifier files: {OGLE_IDENT_GLOB}')
else:
    ident_input = IDENT_PATH
    if not Path(ident_input).exists():
        raise FileNotFoundError(ident_input)

if not PHOT_DIR.exists():
    raise FileNotFoundError(PHOT_DIR)

df_ident, ls_data = data_loader(
    ident_input, PHOT_DIR, mode=MODE, max_workers=N_FIT_WORKERS)
ids = list(ls_data.keys())
print(f'Loaded {len(ids):,} sources')


In [ ]:
# -----------------------------------------------------------------------------
# 3.2 Optional RRFit objects and module wiring
# -----------------------------------------------------------------------------
df_rrfit = None       # Replace with the loaded RRFit result if INIT='rrfit'.
templates = None      # Replace with the RRFit template dictionary if needed.

if INIT == 'rrfit' and (df_rrfit is None or templates is None):
    raise ValueError("INIT='rrfit' requires df_rrfit and templates")

wire_globals(
    decomp, ls_data, df_ident, df_rrfit=df_rrfit, templates=templates)
print('decomposition globals wired')


In [ ]:
# -----------------------------------------------------------------------------
# 3.3 Small I/O helpers
# -----------------------------------------------------------------------------
def row_to_record(row, mode=MODE):
    if row is None:
        raise RuntimeError('Fourier decomposition returned None')
    return dict(zip(build_fd_header(mode), row))


def read_fd_table(path, mode=MODE):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, sep=r'\s+', dtype={'ID': str})
    frame['ID_key'] = frame['ID'].astype(str)
    return frame


def _jsonable(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, tuple):
        return list(value)
    raise TypeError(f'Not JSON serializable: {type(value)}')


def save_json_atomic(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, default=_jsonable, ensure_ascii=False)
    temporary.replace(path)


## 4. Single-source fit

Always run this section before enabling catalog multiprocessing. Check the period, order, final $\chi^2$, phase gaps, and the plotted light curve.


In [ ]:
# -----------------------------------------------------------------------------
# 4.1 Single fit
# -----------------------------------------------------------------------------
SINGLE_ID = ids[0]   # Replace with a source of interest.
single_row = decomp.fourier_decomp(
    SINGLE_ID, mode=MODE, verbose=True, **FIT_KWARGS)
single_record = row_to_record(single_row)

display(pd.DataFrame([single_record])[
    ['ID', 'pulsation', 'P0', 'P', 'M_fit', 'chi2', 'fobj', 'flag']])


In [ ]:
# -----------------------------------------------------------------------------
# 4.2 Joint conditional covariance for the shared multi-band morphology
# -----------------------------------------------------------------------------
cfg = get_data_config(MODE)
t, mag, emag, bands = epoch_arrays(ls_data, SINGLE_ID, mode=MODE)
PRIMARY_BAND = str(cfg.filters[cfg.activated_bands[0]])
ACTIVE_BANDS = [str(cfg.filters[i]) for i in cfg.activated_bands]

conditional_single = conditional_shared_invariant_uncertainty(
    t, mag, emag, bands, nominal_record=single_record,
    selected_filters=ACTIVE_BANDS, reference_band=PRIMARY_BAND,
    P=float(single_record['P']), E=float(single_record['E']),
    M_fit=int(single_record['M_fit']),
    n_draws=4000, random_state=RANDOM_STATE, robust=True,
)

display(pd.DataFrame({
    'value': [conditional_single['R21']['median'], conditional_single['R31']['median']],
    'robust_sigma': [conditional_single['R21']['robust_sigma'], conditional_single['R31']['robust_sigma']],
}, index=['R21', 'R31']))
print('condition number:', conditional_single['conditional_fit']['condition_number'])
print('max leverage:', conditional_single['conditional_fit']['max_leverage'])


In [ ]:
# -----------------------------------------------------------------------------
# 4.3 Joint-HC3 confidence bands with shared multi-band morphology
# -----------------------------------------------------------------------------
# This is a confidence interval for the fitted mean curve conditional on
# fixed P, E, and M_fit. It is not a noisy-epoch prediction interval and
# does not include period aliases or order/regularization selection.
fig, axes, conditional_curve_single = plot_lc_conditional_ci(
    SINGLE_ID,
    P=float(single_record['P']),
    E=float(single_record['E']),
    M_fit=int(single_record['M_fit']),
    mode=MODE,
    selected_filters=ACTIVE_BANDS,
    phase_max=2,
    n_grid=400,
    n_draws=4000,
    random_state=RANDOM_STATE,
    robust=True,
    epoch_data=(t, mag, emag, bands),
    nominal_record=single_record,
    shared_conditional_fit=conditional_single['conditional_fit'],
)


## 5. Catalog multiprocessing

`mp_run` parallelizes over sources and uses a single writer. Existing IDs are excluded before appending, so rerunning the cell does not intentionally duplicate completed rows. On Linux use `fork`; on Windows use `spawn`.


In [ ]:
# -----------------------------------------------------------------------------
# 5.1 Resume-safe source list
# -----------------------------------------------------------------------------
if FD_OUTPUT.exists():
    completed = set(read_fd_table(FD_OUTPUT)['ID_key'])
else:
    completed = set()

ids_todo = [sid for sid in ids if str(sid) not in completed]
print(f'completed={len(completed):,}, remaining={len(ids_todo):,}')


In [ ]:
# -----------------------------------------------------------------------------
# 5.2 Multiprocessing Fourier decomposition
# -----------------------------------------------------------------------------
if RUN_CATALOG_FIT:
    mp_run(
        FD_OUTPUT, ids_todo, mode=MODE, max_workers=N_FIT_WORKERS,
        chunksize=32, mp_context=MP_CONTEXT, verbose=False, **FIT_KWARGS,
    )
else:
    print('Dry run. Set RUN_CATALOG_FIT=True after validating the single fit.')


## 6. Full-pipeline bootstrap for one source

The formal default is 100 replicates. Gaia G/BP/RP rows are resampled together by `transit_id` when available. Failures, period changes, and order changes are retained as diagnostics rather than silently discarded.


In [ ]:
# -----------------------------------------------------------------------------
# 6.1 Single-source bootstrap
# -----------------------------------------------------------------------------
N_BOOT = 100
BOOTSTRAP_ID = SINGLE_ID

if RUN_SINGLE_BOOTSTRAP:
    boot_single = bootstrap_fourier_decomp(
        BOOTSTRAP_ID, mode=MODE, n_boot=N_BOOT,
        random_state=RANDOM_STATE, decomp_kwargs=FIT_KWARGS,
        nominal_row=single_row if BOOTSTRAP_ID == SINGLE_ID else None,
        return_replicates=True,
        progress=lambda done, total: print(
            f'bootstrap {done}/{total}', end='\r') if done % 10 == 0 else None,
    )
    save_json_atomic(
        boot_single, BOOTSTRAP_DIR / f'bootstrap_{BOOTSTRAP_ID}.json')
    print('\nBootstrap saved')
else:
    print('Set RUN_SINGLE_BOOTSTRAP=True to run the formal bootstrap.')


In [ ]:
# -----------------------------------------------------------------------------
# 6.2 Bootstrap summary
# -----------------------------------------------------------------------------
if 'boot_single' in globals():
    inv = boot_single['invariants']
    display(pd.DataFrame({
        'value': [inv['R21']['median'], inv['R31']['median'],
                  inv['phi21']['mean'], inv['phi31']['mean']],
        'robust_sigma': [inv['R21']['robust_sigma'], inv['R31']['robust_sigma'],
                         inv['phi21']['robust_sigma'], inv['phi31']['robust_sigma']],
        'valid_fraction': [1.0, 1.0, inv['phi21_valid_fraction'],
                           inv['phi31_valid_fraction']],
    }, index=['R21', 'R31', 'phi21', 'phi31']))
    print('failure fraction:', boot_single['failure_fraction'])
    print('nominal-period fraction:', boot_single['period_nominal_fraction'])
    print('M_fit probabilities:', boot_single['m_fit_probability'])


## 7. Phase-folded light curve with bootstrap confidence intervals

The default `period_mode='nominal'` excludes half/double/other period solutions from the morphology band and reports their fraction separately. Use `period_mode='all'` to visualize total period-selection uncertainty.


In [ ]:
# -----------------------------------------------------------------------------
# 7.1 Confidence-band plotting helpers
# -----------------------------------------------------------------------------
def evaluate_record(record, band, t_eval):
    m_fit = int(record['M_fit'])
    A = np.asarray([record[f'A{k}'] for k in range(1, m_fit + 1)], dtype=float)
    Q = np.asarray([record[f'Q{k}'] for k in range(1, m_fit + 1)], dtype=float)
    theta = np.array([record[f'm0_{band}'], record[f'amp_{band}'],
                      A, Q, record['P'], record['E']], dtype=object)
    return F(theta, np.asarray(t_eval), M_fit=m_fit, coef_mode='AQ')


def plot_lightcurve_with_ci(sid, nominal, bootstrap_result, epoch_data,
                            mode=MODE, period_mode='nominal', n_grid=300,
                            savepath=None):
    cfg = get_data_config(mode)
    t, mag, emag, bands = epoch_data
    P0, E0 = float(nominal['P']), float(nominal['E'])
    phase_grid = np.linspace(0.0, 1.0, n_grid, endpoint=False)
    t_grid = E0 + P0 * phase_grid

    replicates = bootstrap_result.get('replicates', [])
    if period_mode == 'nominal':
        replicates = [r for r in replicates
                      if np.isclose(float(r['P']), P0, rtol=1e-3)]
    elif period_mode != 'all':
        raise ValueError("period_mode must be 'nominal' or 'all'")

    filters = [str(b) for b in cfg.filters if np.any(bands == b)]
    fig, axes = plt.subplots(len(filters), 1, figsize=(8, 2.8 * len(filters)),
                             sharex=True, squeeze=False)
    axes = axes[:, 0]

    for ax, band in zip(axes, filters):
        observed = bands == band
        phase_obs = ((t[observed] - E0) / P0) % 1.0
        phase_obs2 = np.r_[phase_obs, phase_obs + 1.0]
        mag_obs2 = np.r_[mag[observed], mag[observed]]
        err_obs2 = np.r_[emag[observed], emag[observed]]
        ax.errorbar(phase_obs2, mag_obs2, yerr=err_obs2, fmt='.', ms=4,
                    color=cfg.lc_colors[list(cfg.filters).index(band)],
                    alpha=0.65, lw=0.5, label=f'{band} epochs')

        nominal_curve = evaluate_record(nominal, band, t_grid)
        x2 = np.r_[phase_grid, phase_grid + 1.0]
        ax.plot(x2, np.r_[nominal_curve, nominal_curve], color='black',
                lw=1.5, label='nominal fit')

        curves = []
        for record in replicates:
            try:
                curves.append(evaluate_record(record, band, t_grid))
            except (KeyError, ValueError, TypeError):
                continue
        if curves:
            curves = np.asarray(curves)
            q025, q16, q84, q975 = np.nanpercentile(
                curves, [2.5, 16.0, 84.0, 97.5], axis=0)
            ax.fill_between(x2, np.r_[q025, q025], np.r_[q975, q975],
                            color='tab:blue', alpha=0.12, label='95% bootstrap CI')
            ax.fill_between(x2, np.r_[q16, q16], np.r_[q84, q84],
                            color='tab:blue', alpha=0.28, label='68% bootstrap CI')

        ax.set_ylabel(f'{band} [mag]')
        ax.invert_yaxis()
        ax.grid(alpha=0.15)
        ax.legend(loc='best', fontsize=8, ncol=2)

    axes[-1].set_xlabel('Phase')
    axes[-1].set_xlim(0.0, 2.0)
    title = (f'{sid} | P={P0:.6f} d | M={int(nominal["M_fit"])} | '
             f'bootstrap curves={len(replicates)} | mode={period_mode}')
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    if savepath is not None:
        fig.savefig(savepath, dpi=200, bbox_inches='tight')
    return fig, axes


In [ ]:
# -----------------------------------------------------------------------------
# 7.2 Plot the selected source
# -----------------------------------------------------------------------------
if 'boot_single' in globals():
    epoch_single = epoch_arrays(ls_data, BOOTSTRAP_ID, mode=MODE)
    nominal_for_plot = (single_record if BOOTSTRAP_ID == SINGLE_ID else
                        row_to_record(decomp.fourier_decomp(
                            BOOTSTRAP_ID, mode=MODE, verbose=False, **FIT_KWARGS)))
    plot_lightcurve_with_ci(
        BOOTSTRAP_ID, nominal_for_plot, boot_single, epoch_single,
        period_mode='nominal',
        savepath=FIGURE_DIR / f'lc_ci_{BOOTSTRAP_ID}.png')
    plt.show()
else:
    print('Run the single-source bootstrap first.')


## 8. Optional multiprocessing bootstrap

This section is intended for a Linux CPU server using `fork`. It parallelizes over sources; each worker runs its source's bootstrap replicates serially. Each source is checkpointed to an independent JSON file.


In [ ]:
# -----------------------------------------------------------------------------
# 8.1 Batch bootstrap worker
# -----------------------------------------------------------------------------
BOOTSTRAP_IDS = ids[:100]   # Replace with a predeclared validation subset.

def _bootstrap_worker(sid):
    output = BOOTSTRAP_DIR / f'bootstrap_{sid}.json'
    if output.exists():
        return sid, 'exists', None
    try:
        result = bootstrap_fourier_decomp(
            sid, mode=MODE, n_boot=N_BOOT,
            random_state=RANDOM_STATE, decomp_kwargs=FIT_KWARGS,
            return_replicates=False)
        save_json_atomic(result, output)
        return sid, 'ok', None
    except Exception as exc:
        return sid, 'failed', repr(exc)


if RUN_BOOTSTRAP_BATCH:
    if N_BOOT_WORKERS > 1 and MP_CONTEXT != 'fork':
        raise RuntimeError(
            'Notebook multiprocessing bootstrap requires Linux/fork. '
            'Use N_BOOT_WORKERS=1 on spawn-based systems.')

    if N_BOOT_WORKERS == 1:
        batch_status = [_bootstrap_worker(sid) for sid in tqdm(BOOTSTRAP_IDS)]
    else:
        ctx = mp.get_context(MP_CONTEXT)
        with ctx.Pool(processes=N_BOOT_WORKERS, maxtasksperchild=5) as pool:
            iterator = pool.imap_unordered(_bootstrap_worker, BOOTSTRAP_IDS, chunksize=1)
            batch_status = list(tqdm(iterator, total=len(BOOTSTRAP_IDS)))
    display(pd.DataFrame(batch_status, columns=['ID', 'status', 'error'])
            .groupby('status').size())
else:
    print('Dry run. Set RUN_BOOTSTRAP_BATCH=True after fixing BOOTSTRAP_IDS.')


## 9. Build an uncertainty table

The flattened table keeps fit failure, period stability, order stability, harmonic S/N, amplitude-ratio uncertainty, and circular phase uncertainty. These are measurement/fitting uncertainties and must remain separate from ML ensemble scatter.


In [ ]:
# -----------------------------------------------------------------------------
# 9.1 Flatten bootstrap JSON summaries
# -----------------------------------------------------------------------------
def flatten_bootstrap(result):
    period = result['period']
    inv = result['invariants']
    harmonics = result.get('harmonics', {})
    row = {
        'ID_key': str(result['sid']),
        'n_boot': result['n_boot'],
        'n_success': result['n_success'],
        'failure_fraction': result['failure_fraction'],
        'P_med': period['median'], 'P_q16': period['q16'], 'P_q84': period['q84'],
        'period_nominal_fraction': result['period_nominal_fraction'],
        'period_half_double_fraction': result['period_half_double_fraction'],
        'm_fit_entropy': result['m_fit_entropy'],
        'p_m_ge_2': result['p_m_ge_2'], 'p_m_ge_3': result['p_m_ge_3'],
        'R21': inv['R21']['median'], 'R21_q16': inv['R21']['q16'],
        'R21_q84': inv['R21']['q84'], 'R21_sigma': inv['R21']['robust_sigma'],
        'R31': inv['R31']['median'], 'R31_q16': inv['R31']['q16'],
        'R31_q84': inv['R31']['q84'], 'R31_sigma': inv['R31']['robust_sigma'],
        'phi21': inv['phi21']['mean'], 'phi21_sigma': inv['phi21']['robust_sigma'],
        'phi31': inv['phi31']['mean'], 'phi31_sigma': inv['phi31']['robust_sigma'],
        'phi21_valid_fraction': inv['phi21_valid_fraction'],
        'phi31_valid_fraction': inv['phi31_valid_fraction'],
    }
    for k in range(1, 4):
        stats = harmonics.get(f'A{k}', {})
        row[f'A{k}_sigma'] = stats.get('robust_sigma', np.nan)
        row[f'A{k}_snr'] = stats.get('robust_snr', np.nan)
    return row


summary_rows = []
for path in sorted(BOOTSTRAP_DIR.glob('bootstrap_*.json')):
    with path.open(encoding='utf-8') as handle:
        summary_rows.append(flatten_bootstrap(json.load(handle)))

df_unc = pd.DataFrame(summary_rows)
if len(df_unc):
    df_unc['logP'] = np.log10(df_unc['P_med'])
    df_unc['logP_err_lo'] = df_unc['logP'] - np.log10(df_unc['P_q16'])
    df_unc['logP_err_hi'] = np.log10(df_unc['P_q84']) - df_unc['logP']

    if FD_OUTPUT.exists():
        df_fd = read_fd_table(FD_OUTPUT)
        keep = ['ID_key', 'pulsation', 'M_fit', 'chi2', 'flag']
        df_unc = df_unc.merge(df_fd[keep], on='ID_key', how='left')
display(df_unc.head())


## 10. $\log P$--Fourier space with error bars

Amplitude-ratio intervals are asymmetric 16--84 percentile ranges. Phase errors use circular robust sigma; error segments crossing 0 or $2\pi$ are split at the boundary. Large phase uncertainty or low valid fraction should be treated as a validity failure, especially for $\phi_{31}$.


In [ ]:
# -----------------------------------------------------------------------------
# 10.1 Circular error-bar helper
# -----------------------------------------------------------------------------
def circular_errorbar(ax, x, y, xerr_lo, xerr_hi, yerr, color, label=None,
                      alpha=0.45, markersize=3):
    x = np.asarray(x, dtype=float)
    y = np.mod(np.asarray(y, dtype=float), 2 * np.pi)
    yerr = np.asarray(yerr, dtype=float)
    xerr_lo = np.asarray(xerr_lo, dtype=float)
    xerr_hi = np.asarray(xerr_hi, dtype=float)
    good = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr)
    ax.scatter(x[good], y[good], s=markersize**2, color=color, alpha=alpha,
               label=label, zorder=3)
    for xi, yi, xlo, xhi, ei in zip(
            x[good], y[good], xerr_lo[good], xerr_hi[good], yerr[good]):
        ax.hlines(yi, xi - xlo, xi + xhi, color=color, alpha=0.25, lw=0.6)
        if ei >= np.pi:
            ax.vlines(xi, 0.0, 2 * np.pi, color=color, alpha=0.25, lw=0.6)
        elif yi - ei < 0.0:
            ax.vlines(xi, 0.0, yi + ei, color=color, alpha=0.25, lw=0.6)
            ax.vlines(xi, yi - ei + 2 * np.pi, 2 * np.pi,
                      color=color, alpha=0.25, lw=0.6)
        elif yi + ei > 2 * np.pi:
            ax.vlines(xi, yi - ei, 2 * np.pi, color=color, alpha=0.25, lw=0.6)
            ax.vlines(xi, 0.0, yi + ei - 2 * np.pi,
                      color=color, alpha=0.25, lw=0.6)
        else:
            ax.vlines(xi, yi - ei, yi + ei, color=color, alpha=0.25, lw=0.6)


def plot_logp_fourier_space(frame, class_col='pulsation', savepath=None):
    required = {'logP', 'R21', 'R31', 'phi21', 'phi31'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')

    fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True)
    classes = (frame[class_col].fillna('unknown').unique()
               if class_col in frame else np.array(['all']))
    cmap = plt.get_cmap('tab10')

    for i, class_name in enumerate(classes):
        subset = frame if class_col not in frame else frame[frame[class_col].fillna('unknown') == class_name]
        color = cmap(i % 10)
        x = subset['logP'].to_numpy(float)
        xerr = np.vstack([subset['logP_err_lo'], subset['logP_err_hi']])

        for ax, name in zip(axes[0], ['R21', 'R31']):
            y = subset[name].to_numpy(float)
            ylo = y - subset[f'{name}_q16'].to_numpy(float)
            yhi = subset[f'{name}_q84'].to_numpy(float) - y
            good = np.isfinite(x) & np.isfinite(y) & np.isfinite(ylo) & np.isfinite(yhi)
            ax.errorbar(x[good], y[good], xerr=xerr[:, good],
                        yerr=np.vstack([ylo[good], yhi[good]]), fmt='o', ms=3,
                        color=color, alpha=0.40, elinewidth=0.5, capsize=0,
                        label=str(class_name))

        for ax, name in zip(axes[1], ['phi21', 'phi31']):
            circular_errorbar(
                ax, x, subset[name], subset['logP_err_lo'], subset['logP_err_hi'],
                subset[f'{name}_sigma'], color=color, label=str(class_name))

    axes[0, 0].set_ylabel(r'$R_{21}$')
    axes[0, 1].set_ylabel(r'$R_{31}$')
    axes[1, 0].set_ylabel(r'$\phi_{21}$ [rad]')
    axes[1, 1].set_ylabel(r'$\phi_{31}$ [rad]')
    for ax in axes[1]:
        ax.set_ylim(0.0, 2 * np.pi)
        ax.set_yticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
        ax.set_yticklabels(['0', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'])
    for ax in axes.flat:
        ax.grid(alpha=0.15)
        ax.set_xlabel(r'$\log_{10}(P/\mathrm{day})$')
    axes[0, 0].legend(fontsize=8, markerscale=1.5, ncol=2)
    fig.tight_layout()
    if savepath is not None:
        fig.savefig(savepath, dpi=220, bbox_inches='tight')
    return fig, axes


In [ ]:
# -----------------------------------------------------------------------------
# 10.2 Reliability-filtered diagnostic plot
# -----------------------------------------------------------------------------
if len(df_unc):
    # Exploratory plotting filter, not a Gold/calibration threshold.
    plot_mask = (
        (df_unc['failure_fraction'] <= 0.20) &
        (df_unc['period_nominal_fraction'] >= 0.80) &
        (df_unc['phi21_valid_fraction'] >= 0.80)
    )
    df_plot = df_unc.loc[plot_mask].copy()
    print(f'plotting {len(df_plot):,}/{len(df_unc):,} uncertainty-qualified sources')
    plot_logp_fourier_space(
        df_plot, savepath=FIGURE_DIR / f'logP_fourier_errorbar_{MODE}.png')
    plt.show()
else:
    print('No bootstrap summaries were found.')


## 11. Validation checklist

Before a catalog-scale run, verify:

- the single-source period is not an LS/window/harmonic alias,
- the final curve and confidence band follow the observed steep branch without gap oscillation,
- `failure_fraction`, `period_nominal_fraction`, and `m_fit_probability` are retained,
- $\phi_{31}$ is masked when $A_3$ support or circular validity is poor,
- conditional covariance is treated as a lower bound,
- bootstrap measurement/fitting uncertainty is not combined with ML ensemble scatter, and
- old `use_optim=True` Fourier tables are not mixed with corrected outputs without provenance auditing.
